In [1]:
import os

In [2]:
%pwd

'd:\\Siam\\End-to-End-ML-Project-with-MLflow\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'd:\\Siam\\End-to-End-ML-Project-with-MLflow'

In [5]:
from pathlib import Path
from dataclasses import dataclass

In [6]:
@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir: Path
    train_data_path: Path
    test_data_path: Path
    model_name: str
    alpha: float
    l1_ratio: float
    target_column: str

In [7]:
from mlProject.constants import *
from mlProject.utils.common import read_yaml, create_directories

In [8]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config["artifacts_root"]])


    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config = self.config["model_trainer"]
        params = self.params["ElasticNet"]
        schema = self.schema["TARGET_COLUMN"]

        create_directories([config["root_dir"]])

        model_trainer_config = ModelTrainerConfig(
            root_dir = config["root_dir"],
            train_data_path = config["train_data_path"],
            test_data_path = config["test_data_path"],
            model_name = config["model_name"],
            alpha = params["alpha"],
            l1_ratio = params["l1_ratio"],
            target_column = schema["name"]
        )

        return model_trainer_config

In [9]:
import pandas as pd
import os
from mlProject.logger import logging
from mlProject.exception import CustomException
from sklearn.linear_model import ElasticNet
import joblib
import sys

In [10]:
class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config

    def train_model(self):
        train_data = pd.read_csv(self.config.train_data_path)
        test_data = pd.read_csv(self.config.test_data_path)

        X_train = train_data.drop(self.config.target_column, axis=1)
        y_train = train_data[self.config.target_column]
        X_test = test_data.drop(self.config.target_column, axis=1)
        y_test = test_data[self.config.target_column]

        lr = ElasticNet(alpha=self.config.alpha, l1_ratio=self.config.l1_ratio, random_state=42)
        lr.fit(X_train, y_train)

        model_path = os.path.join(self.config.root_dir, self.config.model_name)
        joblib.dump(lr, model_path)
        logging.info(f"Model trained and saved at {model_path}")

In [11]:
try:
    config = ConfigurationManager()
    model_trainer_config = config.get_model_trainer_config()
    model_trainer = ModelTrainer(config=model_trainer_config)
    model_trainer.train_model()
except Exception as e:
    raise CustomException(e, sys)  

[2026-04-12 19:21:12,646: INFO: common: yaml file: D:\Siam\End-to-End-ML-Project-with-MLflow\config\config.yaml loaded successfully]
[2026-04-12 19:21:12,649: INFO: common: yaml file: D:\Siam\End-to-End-ML-Project-with-MLflow\params.yaml loaded successfully]
[2026-04-12 19:21:12,652: INFO: common: yaml file: D:\Siam\End-to-End-ML-Project-with-MLflow\schema.yaml loaded successfully]
[2026-04-12 19:21:12,653: INFO: common: created directory at: artifacts]
[2026-04-12 19:21:12,655: INFO: common: created directory at: artifacts/model_trainer]
[2026-04-12 19:21:12,668: INFO: 1566592567: Model trained and saved at artifacts/model_trainer\model.joblib]
